# Exploratory Data Analysis: Atypical Presentations of Rheumatoid Arthritis

**Purpose:** reproducible, leakage-aware EDA for four separate binary outcomes:

- `NegRFandACCP`
- `NoHandJointInvolvement`
- `AsymmetricArthritis`
- `OnlyLargeJoints`

This notebook:

1. validates the source CSV without modifying it;
2. audits types, missingness, distributions, rarity, redundancy, and outcome associations;
3. performs explicit target-definition/leakage checks;
4. creates primary and sensitivity datasets for each outcome;
5. saves all tables and plots;
6. converts a freshly executed copy of the notebook to a self-contained HTML report in the final cell.

> **Important:** This is descriptive EDA, not model training. Association rankings are not causal findings and are not used for feature selection outside a future resampling pipeline.


In [ ]:
# 1. Imports and dependency check
from __future__ import annotations

import json
import math
import os
import platform
import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="colorblind",
    rc={
        "figure.facecolor": "#111827",
        "axes.facecolor": "#172033",
        "axes.edgecolor": "#94a3b8",
        "axes.labelcolor": "#f1f5f9",
        "axes.titlecolor": "#f8fafc",
        "xtick.color": "#e2e8f0",
        "ytick.color": "#e2e8f0",
        "text.color": "#f8fafc",
        "grid.color": "#334155",
        "legend.facecolor": "#1f2937",
        "legend.edgecolor": "#64748b",
    },
)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("seaborn:", sns.__version__)


In [ ]:
# 2. Project configuration and path resolution
CSV_NAME = "Atypical_presentations_RA_predictors_valid_c_exclusions_final_population.csv"
NOTEBOOK_NAME = "RA_EDA.ipynb"
ENV_PROJECT = Path(os.environ['RA_PROJECT_DIR']).expanduser() if os.environ.get('RA_PROJECT_DIR') else Path.cwd()

# On the investigator's Windows machine, the first path is the intended source.
# Additional candidates make the notebook portable for review and verification.
candidate_data_paths = [
    ENV_PROJECT / CSV_NAME,
    Path.cwd() / CSV_NAME,
    Path.cwd() / "uploads" / CSV_NAME,
    Path(os.environ["RA_DATA_PATH"]).expanduser() if os.environ.get("RA_DATA_PATH") else Path.cwd() / "uploads" / CSV_NAME,
]
DATA_PATH = next((p for p in candidate_data_paths if p.exists()), None)
if DATA_PATH is None:
    attempted = "\n".join(f"  - {p}" for p in candidate_data_paths)
    raise FileNotFoundError(f"Could not find {CSV_NAME}. Paths checked:\n{attempted}")

# Save to the authorized RA_PROJECT_DIR when it exists; otherwise use a local verification directory.
if ENV_PROJECT.exists():
    PROJECT_DIR = ENV_PROJECT
else:
    PROJECT_DIR = Path.cwd() / "ra_eda_outputs"

TABLE_DIR = PROJECT_DIR / "eda_outputs" / "tables"
FIGURE_DIR = PROJECT_DIR / "eda_outputs" / "figures"
REPORT_DIR = PROJECT_DIR / "eda_outputs" / "report"
DERIVED_DIR = PROJECT_DIR / "derived_data"
for directory in (TABLE_DIR, FIGURE_DIR, REPORT_DIR, DERIVED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

OUTCOMES = [
    "NegRFandACCP",
    "NoHandJointInvolvement",
    "AsymmetricArthritis",
    "OnlyLargeJoints",
]
IDENTIFIER = "VisitID"
CONSTANT_DESIGN_FLAGS = ["NowPregnant", "Cancer", "NotCurrentlyClassifiableasRA2010"]
REPRODUCTIVE_VARIABLES = [
    "Menopause", "PregnancyHistory", "NoOfPregnancy",
    "PregnancyAgeOf1thPregnancy", "FetalLoss", "ContraceptionMethodOCP",
    "ContraceptionMethodIUD", "PrematureInfants",
]

print("Source CSV:", DATA_PATH.resolve())
print("Project output directory:", PROJECT_DIR.resolve())
print("Tables:", TABLE_DIR.resolve())
print("Figures:", FIGURE_DIR.resolve())
print("Derived datasets:", DERIVED_DIR.resolve())


In [ ]:
# 3. Utility functions
TABLE_REGISTRY: list[dict] = []
FIGURE_REGISTRY: list[dict] = []


def safe_name(text: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in str(text)).strip("_")


def save_table(frame: pd.DataFrame, name: str, description: str = "") -> Path:
    path = TABLE_DIR / f"{safe_name(name)}.csv"
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    TABLE_REGISTRY.append({"name": name, "path": str(path), "description": description,
                           "rows": len(frame), "columns": len(frame.columns)})
    return path


def finish_plot(name: str, description: str = "", dpi: int = 180) -> Path:
    path = FIGURE_DIR / f"{safe_name(name)}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight", facecolor=plt.gcf().get_facecolor())
    FIGURE_REGISTRY.append({"name": name, "path": str(path), "description": description})
    plt.show()
    plt.close()
    return path


def semantic_type(series: pd.Series) -> str:
    nonmissing = series.dropna()
    unique = set(nonmissing.unique().tolist())
    if pd.api.types.is_object_dtype(series) or isinstance(series.dtype, pd.CategoricalDtype):
        return "categorical"
    if unique and unique.issubset({0, 1, 0.0, 1.0}):
        return "constant_binary" if len(unique) == 1 else "binary"
    if len(unique) == 0:
        return "all_missing"
    return "numeric_count_or_continuous"


def wilson_interval(successes: int, total: int, confidence: float = 0.95) -> tuple[float, float]:
    if total == 0:
        return (np.nan, np.nan)
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    p = successes / total
    denominator = 1 + z**2 / total
    centre = (p + z**2 / (2 * total)) / denominator
    half = z * math.sqrt((p * (1 - p) / total) + z**2 / (4 * total**2)) / denominator
    return centre - half, centre + half


def cramers_v(x: pd.Series, y: pd.Series) -> float:
    valid = x.notna() & y.notna()
    table = pd.crosstab(x[valid], y[valid])
    if table.shape[0] < 2 or table.shape[1] < 2 or table.to_numpy().sum() == 0:
        return np.nan
    chi2 = stats.chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()
    phi2 = chi2 / n
    r, k = table.shape
    # Bias-corrected Cramer's V.
    phi2_corr = max(0, phi2 - ((k - 1) * (r - 1)) / max(n - 1, 1))
    r_corr = r - ((r - 1) ** 2) / max(n - 1, 1)
    k_corr = k - ((k - 1) ** 2) / max(n - 1, 1)
    denominator = min(k_corr - 1, r_corr - 1)
    return math.sqrt(phi2_corr / denominator) if denominator > 0 else np.nan


def binary_phi(x: pd.Series, y: pd.Series) -> float:
    valid = x.notna() & y.notna()
    if valid.sum() < 3 or x[valid].nunique() < 2 or y[valid].nunique() < 2:
        return np.nan
    return float(np.corrcoef(x[valid].astype(float), y[valid].astype(float))[0, 1])


def standardized_mean_difference(x: pd.Series, y: pd.Series) -> float:
    valid = x.notna() & y.notna()
    x0 = x[valid & (y == 0)].astype(float)
    x1 = x[valid & (y == 1)].astype(float)
    if len(x0) < 2 or len(x1) < 2:
        return np.nan
    pooled = math.sqrt((x0.var(ddof=1) + x1.var(ddof=1)) / 2)
    return (x1.mean() - x0.mean()) / pooled if pooled > 0 else np.nan


def outcome_association(x: pd.Series, y: pd.Series) -> tuple[str, float, int]:
    valid = x.notna() & y.notna()
    n = int(valid.sum())
    kind = semantic_type(x)
    if n < 3 or y[valid].nunique() < 2:
        return "not_estimable", np.nan, n
    if kind in {"binary", "constant_binary"}:
        return "phi", binary_phi(x, y), n
    if kind == "categorical":
        return "cramers_v", cramers_v(x, y), n
    if x[valid].nunique() < 2:
        return "not_estimable", np.nan, n
    rho = stats.spearmanr(x[valid].astype(float), y[valid].astype(float)).statistic
    return "spearman_rho", float(rho), n


## Data loading and integrity validation

In [ ]:
# 4. Load the source exactly once and validate core structure
df = pd.read_csv(DATA_PATH)

required = set(OUTCOMES + [IDENTIFIER])
missing_required = sorted(required - set(df.columns))
if missing_required:
    raise ValueError(f"Required columns are missing: {missing_required}")

if len(df) == 0:
    raise ValueError("The CSV contains no rows.")
if df.columns.duplicated().any():
    raise ValueError(f"Duplicated column names: {df.columns[df.columns.duplicated()].tolist()}")

source_summary = pd.DataFrame({
    "measure": ["rows", "columns", "unique_VisitID", "missing_VisitID", "exact_duplicate_rows"],
    "value": [len(df), df.shape[1], df[IDENTIFIER].nunique(dropna=True),
              int(df[IDENTIFIER].isna().sum()), int(df.duplicated().sum())],
})
save_table(source_summary, "00_source_summary", "Core source-file integrity checks")
display(source_summary)

# Validate outcome coding without silently coercing unexpected values.
outcome_value_audit = []
for outcome in OUTCOMES:
    observed = sorted(df[outcome].dropna().unique().tolist())
    valid_binary = set(observed).issubset({0, 1, 0.0, 1.0})
    outcome_value_audit.append({"outcome": outcome, "observed_nonmissing_values": str(observed),
                                "valid_binary": valid_binary, "missing_n": int(df[outcome].isna().sum())})
    if not valid_binary:
        raise ValueError(f"{outcome} contains unexpected values: {observed}")
outcome_value_audit = pd.DataFrame(outcome_value_audit)
save_table(outcome_value_audit, "01_outcome_value_audit")
display(outcome_value_audit)


In [ ]:
# 5. Full column inventory and semantic typing
inventory_rows = []
for col in df.columns:
    s = df[col]
    inventory_rows.append({
        "column": col,
        "pandas_dtype": str(s.dtype),
        "semantic_type": semantic_type(s),
        "nonmissing_n": int(s.notna().sum()),
        "missing_n": int(s.isna().sum()),
        "missing_pct": 100 * s.isna().mean(),
        "unique_nonmissing_n": int(s.nunique(dropna=True)),
        "observed_min": s.min() if pd.api.types.is_numeric_dtype(s) and s.notna().any() else np.nan,
        "observed_max": s.max() if pd.api.types.is_numeric_dtype(s) and s.notna().any() else np.nan,
    })
inventory = pd.DataFrame(inventory_rows)
save_table(inventory, "02_column_inventory", "Storage types, semantic types, ranges, and missingness")
display(inventory)

print("Semantic type counts:")
display(inventory["semantic_type"].value_counts().rename_axis("semantic_type").reset_index(name="n_columns"))


## Outcome prevalence and overlap

In [ ]:
# 6. Outcome class balance with Wilson confidence intervals
balance_rows = []
for outcome in OUTCOMES:
    s = df[outcome]
    n = int(s.notna().sum())
    positive = int((s == 1).sum())
    negative = int((s == 0).sum())
    low, high = wilson_interval(positive, n)
    balance_rows.append({
        "outcome": outcome, "nonmissing_n": n, "positive_n": positive,
        "negative_n": negative, "missing_n": int(s.isna().sum()),
        "positive_pct": 100 * positive / n if n else np.nan,
        "wilson_95_low_pct": 100 * low, "wilson_95_high_pct": 100 * high,
    })
outcome_balance = pd.DataFrame(balance_rows)
save_table(outcome_balance, "03_outcome_balance", "Counts, prevalence, and Wilson intervals")
display(outcome_balance)

plt.figure(figsize=(11, 5.5))
ax = sns.barplot(data=outcome_balance, x="outcome", y="positive_pct", color="#0e7490")
errors = np.vstack([
    outcome_balance["positive_pct"] - outcome_balance["wilson_95_low_pct"],
    outcome_balance["wilson_95_high_pct"] - outcome_balance["positive_pct"],
])
ax.errorbar(np.arange(len(outcome_balance)), outcome_balance["positive_pct"], yerr=errors,
            fmt="none", ecolor="#fbbf24", capsize=5, linewidth=2)
ax.set(title="Positive prevalence of the four atypical RA outcomes",
       xlabel="Outcome", ylabel="Positive prevalence (%)")
ax.tick_params(axis="x", rotation=20)
for i, row in outcome_balance.iterrows():
    ax.text(i, row["positive_pct"] + 1.2, f"{int(row['positive_n'])}/{int(row['nonmissing_n'])}",
            ha="center", color="#f8fafc", fontweight="bold")
finish_plot("01_outcome_prevalence", "Outcome prevalence with 95% Wilson intervals")


In [ ]:
# 7. Outcome overlap (descriptive only; outcomes remain separate modeling targets)
outcome_overlap = pd.crosstab(
    df[OUTCOMES].fillna(-1).astype(int).astype(str).agg("|".join, axis=1),
    columns="subjects"
).reset_index().rename(columns={"row_0": "pattern"}).sort_values("subjects", ascending=False)
save_table(outcome_overlap, "04_outcome_overlap_patterns")
display(outcome_overlap.head(20))

pairwise_outcome_phi = df[OUTCOMES].corr(method="pearson")
save_table(pairwise_outcome_phi.reset_index().rename(columns={"index": "outcome"}), "05_outcome_phi_matrix")
plt.figure(figsize=(7, 6))
sns.heatmap(pairwise_outcome_phi, annot=True, fmt=".2f", vmin=-1, vmax=1, center=0,
            cmap="vlag", square=True, cbar_kws={"label": "Phi correlation"})
plt.title("Pairwise overlap of atypical RA outcomes")
finish_plot("02_outcome_phi_heatmap")


## Missingness analysis

In [ ]:
# 8. Missingness summary
missing_summary = inventory[["column", "missing_n", "missing_pct", "semantic_type"]].copy()
missing_summary = missing_summary.sort_values(["missing_pct", "column"], ascending=[False, True]).reset_index(drop=True)
save_table(missing_summary, "06_missingness_summary")
display(missing_summary.head(40))

plot_missing = missing_summary[missing_summary["missing_n"] > 0].head(40).sort_values("missing_pct")
plt.figure(figsize=(11, max(6, 0.28 * len(plot_missing))))
sns.barplot(data=plot_missing, x="missing_pct", y="column", color="#b45309")
plt.title("Variables with the greatest missingness")
plt.xlabel("Missing values (%)")
plt.ylabel("Variable")
finish_plot("03_missingness_barplot")

# Heatmap of the 30 most incomplete variables; rows sorted by total missingness.
top_missing_cols = missing_summary.loc[missing_summary["missing_n"] > 0, "column"].head(30).tolist()
if top_missing_cols:
    row_order = df[top_missing_cols].isna().sum(axis=1).sort_values(ascending=False).index
    plt.figure(figsize=(15, 9))
    sns.heatmap(df.loc[row_order, top_missing_cols].isna(), cmap=["#164e63", "#f59e0b"],
                cbar=False, yticklabels=False)
    plt.title("Missingness pattern: 30 variables with most missing data")
    plt.xlabel("Variable (teal = observed, amber = missing)")
    plt.ylabel("Subjects sorted by number of missing fields")
    plt.xticks(rotation=75, ha="right")
    finish_plot("04_missingness_heatmap_top30")


In [ ]:
# 9. Missingness differences by outcome class
missing_by_outcome_rows = []
for outcome in OUTCOMES:
    valid_y = df[outcome].notna()
    for col in df.columns:
        if col == outcome or df[col].isna().sum() == 0:
            continue
        m0 = df.loc[valid_y & (df[outcome] == 0), col].isna()
        m1 = df.loc[valid_y & (df[outcome] == 1), col].isna()
        if len(m0) == 0 or len(m1) == 0:
            continue
        missing_by_outcome_rows.append({
            "outcome": outcome, "variable": col,
            "missing_pct_outcome_0": 100 * m0.mean(),
            "missing_pct_outcome_1": 100 * m1.mean(),
            "absolute_difference_pct_points": abs(100 * (m1.mean() - m0.mean())),
            "signed_difference_pct_points": 100 * (m1.mean() - m0.mean()),
        })
missing_by_outcome = pd.DataFrame(missing_by_outcome_rows).sort_values(
    ["outcome", "absolute_difference_pct_points"], ascending=[True, False]
)
save_table(missing_by_outcome, "07_missingness_by_outcome")
for outcome in OUTCOMES:
    print(f"Largest missingness differences for {outcome}")
    display(missing_by_outcome[missing_by_outcome["outcome"] == outcome].head(12))


## Univariate distributions, rarity, and outlier flags

In [ ]:
# 10. Binary-variable prevalence and near-constant flags
binary_cols = inventory.loc[inventory["semantic_type"].isin(["binary", "constant_binary"]), "column"].tolist()
binary_rows = []
for col in binary_cols:
    s = df[col]
    observed_n = int(s.notna().sum())
    positive_n = int((s == 1).sum())
    negative_n = int((s == 0).sum())
    minority_n = min(positive_n, negative_n)
    minority_pct = 100 * minority_n / observed_n if observed_n else np.nan
    binary_rows.append({
        "variable": col, "observed_n": observed_n, "positive_n": positive_n,
        "negative_n": negative_n, "missing_n": int(s.isna().sum()),
        "positive_pct_observed": 100 * positive_n / observed_n if observed_n else np.nan,
        "minority_n": minority_n, "minority_pct_observed": minority_pct,
        "near_constant_flag": bool(minority_n < 10 or (pd.notna(minority_pct) and minority_pct < 1.0)),
    })
binary_prevalence = pd.DataFrame(binary_rows).sort_values(
    ["near_constant_flag", "minority_n", "variable"], ascending=[False, True, True]
)
save_table(binary_prevalence, "08_binary_prevalence_and_rarity")
display(binary_prevalence.head(40))

near_constant = binary_prevalence[binary_prevalence["near_constant_flag"]]
print(f"Near-constant rule flagged {len(near_constant)} binary variables (minority n < 10 or minority < 1%).")


In [ ]:
# 11. Numeric/count/continuous summaries and IQR outlier flags
numeric_cols = inventory.loc[inventory["semantic_type"] == "numeric_count_or_continuous", "column"].tolist()
numeric_cols = [c for c in numeric_cols if c != IDENTIFIER]

summary_rows, outlier_rows = [], []
for col in numeric_cols:
    s = pd.to_numeric(df[col], errors="coerce")
    observed = s.dropna()
    if observed.empty:
        continue
    q1, median, q3 = observed.quantile([0.25, 0.5, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = (observed < lower) | (observed > upper) if iqr > 0 else pd.Series(False, index=observed.index)
    summary_rows.append({
        "variable": col, "n": len(observed), "missing_n": int(s.isna().sum()),
        "mean": observed.mean(), "sd": observed.std(ddof=1), "min": observed.min(),
        "q1": q1, "median": median, "q3": q3, "max": observed.max(),
        "skewness": observed.skew(),
    })
    outlier_rows.append({
        "variable": col, "iqr_lower_fence": lower, "iqr_upper_fence": upper,
        "flagged_n": int(flagged.sum()), "flagged_pct_observed": 100 * flagged.mean(),
    })
numeric_summary = pd.DataFrame(summary_rows)
outlier_summary = pd.DataFrame(outlier_rows).sort_values("flagged_pct_observed", ascending=False)
save_table(numeric_summary, "09_numeric_summary")
save_table(outlier_summary, "10_iqr_outlier_flags",
           "Statistical flags only; values must not be deleted without clinical plausibility review")
display(numeric_summary)
display(outlier_summary)

if numeric_cols:
    ncols = 3
    nrows = math.ceil(len(numeric_cols) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.8 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, col in zip(axes, numeric_cols):
        sns.histplot(df[col], kde=True, ax=ax, color="#0891b2", bins="auto")
        ax.set_title(col)
    for ax in axes[len(numeric_cols):]:
        ax.set_visible(False)
    fig.suptitle("Numeric and count-variable distributions", y=1.01, fontsize=16, fontweight="bold")
    finish_plot("05_numeric_distributions")


In [ ]:
# 12. Categorical distributions
categorical_cols = inventory.loc[inventory["semantic_type"] == "categorical", "column"].tolist()
category_rows = []
for col in categorical_cols:
    counts = df[col].fillna("<MISSING>").value_counts(dropna=False)
    for level, n in counts.items():
        category_rows.append({"variable": col, "level": str(level), "n": int(n), "pct_all_rows": 100 * n / len(df)})
category_counts = pd.DataFrame(category_rows)
save_table(category_counts, "11_categorical_level_counts")
display(category_counts)

if categorical_cols:
    ncols = 2
    nrows = math.ceil(len(categorical_cols) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4.2 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, col in zip(axes, categorical_cols):
        order = df[col].fillna("<MISSING>").value_counts().index
        sns.countplot(y=df[col].fillna("<MISSING>"), order=order, ax=ax, color="#7c3aed")
        ax.set_title(col)
        ax.set_xlabel("Subjects")
        ax.set_ylabel("")
    for ax in axes[len(categorical_cols):]:
        ax.set_visible(False)
    fig.suptitle("Categorical-variable distributions", y=1.01, fontsize=16, fontweight="bold")
    finish_plot("06_categorical_distributions")


## Redundancy and correlation diagnostics

In [ ]:
# 13. Pairwise Spearman correlations among numeric variables
numeric_for_corr = [c for c in df.select_dtypes(include=np.number).columns
                    if c not in [IDENTIFIER] + OUTCOMES and df[c].nunique(dropna=True) > 1]
spearman_corr = df[numeric_for_corr].corr(method="spearman", min_periods=30)
save_table(spearman_corr.reset_index().rename(columns={"index": "variable"}), "12_spearman_correlation_matrix")

pairs = []
for i, c1 in enumerate(spearman_corr.columns):
    for c2 in spearman_corr.columns[i + 1:]:
        value = spearman_corr.loc[c1, c2]
        if pd.notna(value):
            pairs.append({"variable_1": c1, "variable_2": c2, "spearman_rho": value,
                          "absolute_rho": abs(value)})
top_corr_pairs = pd.DataFrame(pairs).sort_values("absolute_rho", ascending=False)
save_table(top_corr_pairs, "13_ranked_spearman_pairs")
display(top_corr_pairs.head(40))

# A readable heatmap of the most connected variables rather than an illegible 80x80 plot.
if len(spearman_corr) > 1:
    connectivity = spearman_corr.abs().where(~np.eye(len(spearman_corr), dtype=bool)).max(axis=1)
    heat_cols = connectivity.sort_values(ascending=False).head(min(30, len(connectivity))).index.tolist()
    plt.figure(figsize=(15, 13))
    sns.heatmap(spearman_corr.loc[heat_cols, heat_cols], cmap="vlag", center=0, vmin=-1, vmax=1,
                square=True, cbar_kws={"label": "Spearman correlation"})
    plt.title("Spearman correlation: 30 variables with strongest pairwise redundancy")
    plt.xticks(rotation=75, ha="right", fontsize=8)
    plt.yticks(fontsize=8)
    finish_plot("07_spearman_redundancy_heatmap")


## Outcome-stratified associations (descriptive, not feature selection)

In [ ]:
# 14. Unadjusted outcome associations across candidate variables
excluded_from_association = set([IDENTIFIER] + OUTCOMES + CONSTANT_DESIGN_FLAGS)
association_rows = []
for outcome in OUTCOMES:
    y = df[outcome]
    for col in df.columns:
        if col in excluded_from_association:
            continue
        method, effect, complete_n = outcome_association(df[col], y)
        association_rows.append({
            "outcome": outcome, "variable": col, "semantic_type": semantic_type(df[col]),
            "association_method": method, "association": effect,
            "absolute_association": abs(effect) if pd.notna(effect) else np.nan,
            "pairwise_complete_n": complete_n,
            "missing_n": int(df[col].isna().sum()),
        })
outcome_associations = pd.DataFrame(association_rows).sort_values(
    ["outcome", "absolute_association"], ascending=[True, False]
)
save_table(outcome_associations, "14_unadjusted_outcome_associations",
           "Phi, bias-corrected Cramer's V, or Spearman rho according to variable type")

for outcome in OUTCOMES:
    top = outcome_associations[outcome_associations["outcome"] == outcome].head(20)
    print(f"Top descriptive associations with {outcome}")
    display(top)
    plot_data = top.dropna(subset=["absolute_association"]).sort_values("absolute_association")
    plt.figure(figsize=(10, 7))
    sns.barplot(data=plot_data, x="absolute_association", y="variable", color="#0f766e")
    plt.title(f"Largest unadjusted associations with {outcome}")
    plt.xlabel("Absolute association magnitude (method depends on variable type)")
    plt.ylabel("Variable")
    finish_plot(f"08_associations_{outcome}",
                "Descriptive ranking only; includes possible target-defining variables")


In [ ]:
# 15. Numeric distributions stratified by outcome and standardized mean differences
smd_rows = []
for outcome in OUTCOMES:
    for col in numeric_cols:
        smd_rows.append({"outcome": outcome, "variable": col,
                         "standardized_mean_difference": standardized_mean_difference(df[col], df[outcome])})
smd_table = pd.DataFrame(smd_rows)
smd_table["absolute_smd"] = smd_table["standardized_mean_difference"].abs()
smd_table = smd_table.sort_values(["outcome", "absolute_smd"], ascending=[True, False])
save_table(smd_table, "15_numeric_standardized_mean_differences")

for outcome in OUTCOMES:
    top_numeric = smd_table[(smd_table["outcome"] == outcome) & smd_table["absolute_smd"].notna()].head(6)["variable"].tolist()
    if not top_numeric:
        continue
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.ravel()
    plot_df = df[df[outcome].notna()].copy()
    plot_df[outcome] = plot_df[outcome].astype(int).astype(str)
    for ax, col in zip(axes, top_numeric):
        sns.boxplot(data=plot_df, x=outcome, y=col, ax=ax, color="#0891b2", showfliers=True)
        sns.stripplot(data=plot_df, x=outcome, y=col, ax=ax, color="#fbbf24", alpha=0.25, size=2)
        ax.set_title(col)
        ax.set_xlabel(f"{outcome} (0/1)")
    for ax in axes[len(top_numeric):]:
        ax.set_visible(False)
    fig.suptitle(f"Numeric variables with largest unadjusted SMDs: {outcome}", y=1.01,
                 fontsize=15, fontweight="bold")
    finish_plot(f"09_numeric_by_{outcome}")


## Target-definition and leakage audit

In [ ]:
# 16. Explicit consistency checks for likely outcome components
leakage_audit_rows = []

# Seronegativity derived from anti-CCP and RF among complete source measurements.
sero_complete = df[["NegRFandACCP", "AntiCCPPosNeg", "SerumRheumatoidFactorPositive"]].notna().all(axis=1)
sero_rule = ((df["AntiCCPPosNeg"] == 0) & (df["SerumRheumatoidFactorPositive"] == 0)).astype(float)
sero_agree_n = int((sero_rule[sero_complete] == df.loc[sero_complete, "NegRFandACCP"]).sum())
leakage_audit_rows.append({
    "audit": "NegRFandACCP versus anti-CCP-negative AND RF-negative rule",
    "complete_n": int(sero_complete.sum()), "agreement_n": sero_agree_n,
    "agreement_pct": 100 * sero_agree_n / sero_complete.sum() if sero_complete.sum() else np.nan,
})
sero_crosstab = pd.crosstab(df.loc[sero_complete, "NegRFandACCP"], sero_rule[sero_complete],
                            rownames=["recorded_NegRFandACCP"], colnames=["derived_seronegative_rule"]).reset_index()
save_table(sero_crosstab, "16_seronegativity_rule_crosstab")

# Simple no-hand rule from available hand markers; disagreement still indicates leakage risk.
hand_cols = ["DIPInvolvementBinary", "HandPIPInvolvementBinary", "MCPInvolvementB", "WristInvolvementBinary"]
hand_complete = df[["NoHandJointInvolvement"] + hand_cols].notna().all(axis=1)
no_hand_rule = (df[hand_cols].sum(axis=1) == 0).astype(float)
hand_agree_n = int((no_hand_rule[hand_complete] == df.loc[hand_complete, "NoHandJointInvolvement"]).sum())
leakage_audit_rows.append({
    "audit": "NoHandJointInvolvement versus no DIP/PIP/MCP/wrist involvement",
    "complete_n": int(hand_complete.sum()), "agreement_n": hand_agree_n,
    "agreement_pct": 100 * hand_agree_n / hand_complete.sum() if hand_complete.sum() else np.nan,
})
hand_crosstab = pd.crosstab(df.loc[hand_complete, "NoHandJointInvolvement"], no_hand_rule[hand_complete],
                            rownames=["recorded_NoHandJointInvolvement"], colnames=["derived_no_hand_rule"]).reset_index()
save_table(hand_crosstab, "17_no_hand_rule_crosstab")

# Joint pattern versus asymmetric target.
asym_pattern = pd.crosstab(df["JointInvolvementPattern"].fillna("<MISSING>"),
                           df["AsymmetricArthritis"].fillna(-1), dropna=False).reset_index()
save_table(asym_pattern, "18_asymmetry_by_joint_pattern")

leakage_audit = pd.DataFrame(leakage_audit_rows)
save_table(leakage_audit, "19_target_definition_agreement")
display(leakage_audit)
display(Markdown("**These agreement checks identify definitional overlap and possible coding discrepancies; they do not validate using the source components as primary predictors.**"))


## Outcome-specific datasets and predictor manifests

In [ ]:
# 17. Create leakage-controlled primary and expanded sensitivity datasets
# No values are imputed or otherwise altered here. The only row removal is a missing target.
# VisitID is retained for traceability and explicitly marked DO_NOT_MODEL in each manifest.

TARGET_SPECIFIC_PRIMARY_EXCLUSIONS = {
    "NegRFandACCP": ["AntiCCPPosNeg", "SerumRheumatoidFactorPositive"],
    "NoHandJointInvolvement": [
        "DIPInvolvementBinary", "HandPIPInvolvementBinary", "MCPInvolvementB", "WristInvolvementBinary"
    ],
    "AsymmetricArthritis": ["JointInvolvementPattern"],
    "OnlyLargeJoints": [
        "DASJointCountTEN28", "DASJointCountSW", "PolyarticularType", "JointInvolvementPattern",
        "AnkleInvolvementB", "MTPInvolvementB", "PIPFootInvolvementB",
        "SacroiliacInvolvementB", "KneeInvolvementB", "ElbowInvolvementB",
        "ShoulderInvolvementB", "SpineInvolvementB", "DIPInvolvementBinary",
        "HandPIPInvolvementBinary", "MCPInvolvementB", "WristInvolvementBinary",
    ],
}

manifest_registry = []
for outcome in OUTCOMES:
    other_outcomes = [o for o in OUTCOMES if o != outcome]
    common_exclusions = set(other_outcomes + CONSTANT_DESIGN_FLAGS + REPRODUCTIVE_VARIABLES)
    sensitivity_predictors = [
        c for c in df.columns
        if c not in common_exclusions and c not in {IDENTIFIER, outcome}
    ]
    primary_predictors = [
        c for c in sensitivity_predictors
        if c not in set(TARGET_SPECIFIC_PRIMARY_EXCLUSIONS[outcome])
    ]

    analysis_rows = df[outcome].notna()
    primary_columns = [IDENTIFIER, outcome] + primary_predictors
    sensitivity_columns = [IDENTIFIER, outcome] + sensitivity_predictors
    primary_data = df.loc[analysis_rows, primary_columns].copy()
    sensitivity_data = df.loc[analysis_rows, sensitivity_columns].copy()

    # Assertions protect against accidental inclusion of forbidden columns.
    assert not (set(other_outcomes) & set(primary_data.columns))
    assert not (set(CONSTANT_DESIGN_FLAGS) & set(primary_data.columns))
    assert not (set(REPRODUCTIVE_VARIABLES) & set(primary_data.columns))
    assert not (set(TARGET_SPECIFIC_PRIMARY_EXCLUSIONS[outcome]) & set(primary_predictors))
    assert primary_data[outcome].notna().all()

    primary_path = DERIVED_DIR / f"{outcome}__primary_leakage_controlled.csv"
    sensitivity_path = DERIVED_DIR / f"{outcome}__sensitivity_expanded.csv"
    primary_data.to_csv(primary_path, index=False, encoding="utf-8-sig")
    sensitivity_data.to_csv(sensitivity_path, index=False, encoding="utf-8-sig")

    manifest_rows = []
    for col in sensitivity_columns:
        if col == IDENTIFIER:
            role = "identifier_DO_NOT_MODEL"
        elif col == outcome:
            role = "outcome"
        elif col in primary_predictors:
            role = "primary_and_sensitivity_predictor"
        else:
            role = "sensitivity_only_target_proxy"
        manifest_rows.append({"column": col, "role": role, "semantic_type": semantic_type(df[col]),
                              "missing_n_in_analysis_rows": int(df.loc[analysis_rows, col].isna().sum())})
    manifest = pd.DataFrame(manifest_rows)
    manifest_path = DERIVED_DIR / f"{outcome}__predictor_manifest.csv"
    manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")

    manifest_registry.append({
        "outcome": outcome, "analysis_rows": int(analysis_rows.sum()),
        "primary_predictors_n": len(primary_predictors),
        "sensitivity_predictors_n": len(sensitivity_predictors),
        "primary_dataset": str(primary_path), "sensitivity_dataset": str(sensitivity_path),
        "manifest": str(manifest_path),
        "primary_target_specific_exclusions": "; ".join(TARGET_SPECIFIC_PRIMARY_EXCLUSIONS[outcome]),
    })

dataset_registry = pd.DataFrame(manifest_registry)
save_table(dataset_registry, "20_outcome_specific_dataset_registry")
display(dataset_registry)

with open(DERIVED_DIR / "target_specific_exclusion_rules.json", "w", encoding="utf-8") as f:
    json.dump(TARGET_SPECIFIC_PRIMARY_EXCLUSIONS, f, indent=2, ensure_ascii=False)

readme = "# Outcome-specific derived datasets\\n\\n"
readme += "These files are deterministic column subsets of the source CSV. No predictor imputation or scaling has been applied. "
readme += "Rows with a missing target are removed only for that target. `VisitID` is retained solely for traceability and MUST NOT be modeled.\\n\\n"
readme += "- `__primary_leakage_controlled.csv`: primary predictor scope.\\n"
readme += "- `__sensitivity_expanded.csv`: broader contemporaneous scope; may contain target proxies.\\n"
readme += "- `__predictor_manifest.csv`: role of every included column.\\n"
readme += "- `target_specific_exclusion_rules.json`: explicit primary leakage exclusions.\\n"
(DERIVED_DIR / "README.md").write_text(readme, encoding="utf-8")


## EDA completion summary and saved artifact registry

In [ ]:
# 18. Save registries, environment details, and a machine-readable EDA summary
save_table(pd.DataFrame(TABLE_REGISTRY), "98_table_registry")
save_table(pd.DataFrame(FIGURE_REGISTRY), "99_figure_registry")

eda_summary = {
    "source_csv": str(DATA_PATH.resolve()),
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "unique_visit_ids": int(df[IDENTIFIER].nunique(dropna=True)),
    "duplicate_rows": int(df.duplicated().sum()),
    "outcomes": OUTCOMES,
    "project_output_directory": str(PROJECT_DIR.resolve()),
    "tables_directory": str(TABLE_DIR.resolve()),
    "figures_directory": str(FIGURE_DIR.resolve()),
    "derived_data_directory": str(DERIVED_DIR.resolve()),
    "python_version": sys.version,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "seaborn_version": sns.__version__,
}
with open(PROJECT_DIR / "eda_outputs" / "eda_run_summary.json", "w", encoding="utf-8") as f:
    json.dump(eda_summary, f, indent=2, ensure_ascii=False)

print(f"EDA generated {len(TABLE_REGISTRY)} registered tables and {len(FIGURE_REGISTRY)} registered figures.")
print("All outputs are under:", (PROJECT_DIR / "eda_outputs").resolve())
print("Outcome-specific datasets are under:", DERIVED_DIR.resolve())
display(dataset_registry[["outcome", "analysis_rows", "primary_predictors_n", "sensitivity_predictors_n"]])


## Final cell — execute the complete notebook and export all outputs to HTML

Save this notebook as **`the authorized RA_PROJECT_DIR\RA_EDA.ipynb`** before running the cell below. The cell launches a clean execution of the saved notebook and exports every executed cell output and plot into:

`the authorized RA_PROJECT_DIR\eda_outputs\report\RA_EDA_report.html`

The environment flag prevents recursive HTML export during the clean execution.


In [ ]:
# 19. FINAL CELL: clean execution + self-contained HTML export
if os.environ.get("RA_EDA_EXPORT_MODE") == "1":
    print("HTML export subprocess mode: analysis complete; recursive export intentionally skipped.")
else:
    notebook_candidates = [
        ENV_PROJECT / NOTEBOOK_NAME,
        Path.cwd() / NOTEBOOK_NAME,
        Path.cwd() / "ra_analysis" / NOTEBOOK_NAME,
    ]
    notebook_path = next((p for p in notebook_candidates if p.exists()), None)
    if notebook_path is None:
        checked = "\n".join(f"  - {p}" for p in notebook_candidates)
        raise FileNotFoundError(
            "Save this notebook as RA_EDA.ipynb before exporting. Checked:\n" + checked
        )

    export_env = os.environ.copy()
    export_env["RA_EDA_EXPORT_MODE"] = "1"
    command = [
        sys.executable, "-m", "jupyter", "nbconvert",
        "--to", "html",
        "--execute",
        "--ExecutePreprocessor.timeout=1200",
        "--ExecutePreprocessor.kernel_name=python3",
        "--HTMLExporter.embed_images=True",
        "--output", "RA_EDA_report.html",
        "--output-dir", str(REPORT_DIR),
        str(notebook_path),
    ]
    print("Executing:", " ".join(command))
    completed = subprocess.run(command, env=export_env, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"HTML export failed with exit code {completed.returncode}")

    html_path = REPORT_DIR / "RA_EDA_report.html"
    if not html_path.exists():
        raise FileNotFoundError(f"nbconvert reported success but HTML was not found at {html_path}")
    print("HTML report created successfully:", html_path.resolve())
